# NanoVision: Inference and Evaluation

This notebook demonstrates a typical NanoVision inference and evaluation workflow
using functionality provided by the installed `cnt_project` package.

The notebook intentionally contains orchestration only. Segmentation,
postprocessing, COCO export, and evaluation logic remain implemented inside
`src/cnt_project`.

Workflow:

1. Configure dataset and model locations.
2. Run StarDist inference.
3. Export predictions to COCO format.
4. Evaluate predictions.
5. Inspect generated outputs.

For complete runner arguments, use the corresponding runner with `--help`.

# imports/helper

In [1]:
from utils import run_module
from pathlib import Path

from cnt_project.io.paths import DatasetPaths, OutputPaths



# verify installation

In [ ]:
import cnt_project

print("NanoVision import package:")
print(cnt_project.__file__)

CNTLib import package:
C:\Users\abd93000\PycharmProjects\cnt_project_review\src\cnt_project\__init__.py


## Configuration

Edit the paths and model name below before running inference.

Two inference modes are supported by NanoVision:

- canonical dataset mode using a dataset root + split manifest;
- arbitrary image-folder mode using `--data-dir`.

This example uses the canonical dataset workflow.

In [3]:
# ------------------------------------------------------------------
# User configuration
# ------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "examples":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

DATASET_ROOT = (
    PROJECT_ROOT
    / "data"
    / "cnt_segmentation"
)

MODEL_BASEDIR = (
    PROJECT_ROOT
    / "models"
)

GLOBAL_OUTPUTS_ROOT = (
    PROJECT_ROOT
    / "global_outputs"
)

MODEL_NAME = "batch1_seed_0_64rays_grid2x2"

SUBSET = "test"
SPLIT_NAME = "stratified_seed_0"

RUN_NAME = "cntlib_notebook_inference_example"

# Process the complete subset so that the prediction image set
# matches the frozen ground-truth evaluation subset.
MAX_SAMPLES = None


# ------------------------------------------------------------------
# Resolve canonical paths
# ------------------------------------------------------------------

dataset_paths = DatasetPaths.from_root(
    DATASET_ROOT
)

output_paths = OutputPaths.from_root(
    GLOBAL_OUTPUTS_ROOT
)

DATASET_ROOT = dataset_paths.root
GLOBAL_OUTPUTS_ROOT = output_paths.root

SPLIT_MANIFEST = dataset_paths.split_manifest_csv(
    SPLIT_NAME
)

GT_JSON = dataset_paths.coco_json(
    SPLIT_NAME,
    SUBSET,
    filename="annotations_rle.json",
)

In [4]:
print("Dataset root:")
print(DATASET_ROOT)

print("\nSplit manifest:")
print(SPLIT_MANIFEST)

print("\nModel directory:")
print(MODEL_BASEDIR / MODEL_NAME)

print("\nGlobal outputs root:")
print(GLOBAL_OUTPUTS_ROOT)

print("\nGround-truth COCO JSON:")
print(GT_JSON)


assert DATASET_ROOT.exists(), DATASET_ROOT
assert SPLIT_MANIFEST.exists(), SPLIT_MANIFEST
assert (MODEL_BASEDIR / MODEL_NAME).exists(), (
    MODEL_BASEDIR / MODEL_NAME
)
assert GT_JSON.exists(), GT_JSON

# The output root does not need to exist beforehand.
output_paths.ensure()

print("\nConfiguration paths are valid.")

Dataset root:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation

Split manifest:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\splits\stratified_seed_0.csv

Model directory:
C:\Users\abd93000\PycharmProjects\cnt_project_review\models\batch1_seed_0_64rays_grid2x2

Global outputs root:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs

Ground-truth COCO JSON:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\COCO_mask\stratified_seed_0\test\annotations_rle.json

Configuration paths are valid.


### optional CLI discovery

In [5]:
run_module(
    "cnt_project.model_development.inference.runners.infer_runner",
    "--help",
)

Running:
c:\Users\abd93000\PycharmProjects\cnt_project_review\.venv_review\Scripts\python.exe -u -m cnt_project.model_development.inference.runners.infer_runner --help
--------------------------------------------------------------------------------
2026-09-17 09:17:18.244107: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-17 09:17:18.985484: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated fo

## build inference arguments

In [13]:
inference_args = [
    "--dataset-root",
    str(DATASET_ROOT),

    "--split-manifest-path",
    str(SPLIT_MANIFEST),

    "--subset",
    SUBSET,

    "--model-basedir",
    str(MODEL_BASEDIR),

    "--model-name",
    MODEL_NAME,

    "--run-name",
    RUN_NAME,

    "--global-outputs-root",
    str(GLOBAL_OUTPUTS_ROOT),

    "--export-coco",
    "--export-mask-coco",
    
    "--apply-smoothing",
    "false",
]

if MAX_SAMPLES is not None:
    inference_args.extend(
        [
            "--max-samples",
            str(MAX_SAMPLES),
        ]
    )

inference_args

['--dataset-root',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\data\\cnt_segmentation',
 '--split-manifest-path',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\data\\cnt_segmentation\\splits\\stratified_seed_0.csv',
 '--subset',
 'test',
 '--model-basedir',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\models',
 '--model-name',
 'batch1_seed_0_64rays_grid2x2',
 '--run-name',
 'cntlib_notebook_inference_example',
 '--global-outputs-root',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\global_outputs',
 '--export-coco',
 '--export-mask-coco',
 '--apply-smoothing',
 'false']

### run inference

In [14]:
run_module(
    "cnt_project.model_development.inference.runners.infer_runner",
    *inference_args,
)

Running:
c:\Users\abd93000\PycharmProjects\cnt_project_review\.venv_review\Scripts\python.exe -u -m cnt_project.model_development.inference.runners.infer_runner --dataset-root C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation --split-manifest-path C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\splits\stratified_seed_0.csv --subset test --model-basedir C:\Users\abd93000\PycharmProjects\cnt_project_review\models --model-name batch1_seed_0_64rays_grid2x2 --run-name cntlib_notebook_inference_example --global-outputs-root C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs --export-coco --export-mask-coco --apply-smoothing false
--------------------------------------------------------------------------------
2026-09-17 09:32:06.151351: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders

## Evaluation
NanoVision's current instance-segmentation evaluator supports:
- object-level IoU matching;
- object-level clDice matching;
- normalized centerline-distance matching;
- image-level pixel metrics.
This example evaluates object matching using clDice at a threshold of 0.5,
which is the primary object-matching configuration used in the CNT analysis.
Pixel-level metrics are computed automatically by the evaluator.


In [17]:
run_module(
    "cnt_project.evaluation.runners.eval_instance_metrics_runner",
    "--help",
)

Running:
c:\Users\abd93000\PycharmProjects\cnt_project_review\.venv_review\Scripts\python.exe -u -m cnt_project.evaluation.runners.eval_instance_metrics_runner --help
--------------------------------------------------------------------------------
usage: eval_instance_metrics_runner.py [-h]
                                       [--matching-metrics {iou,cldice,centerline} [{iou,cldice,centerline} ...]]
                                       --gt-rle-json GT_RLE_JSON
                                       --pred-rle-json PRED_RLE_JSON
                                       --output-dir OUTPUT_DIR
                                       [--prediction-score-threshold PREDICTION_SCORE_THRESHOLD]
                                       [--iou-thresholds IOU_THRESHOLDS]
                                       [--cldice-thresholds CLDICE_THRESHOLDS]
                                       [--centerline-distance-measure {chamfer,hausdorff,both}]
                                       [--centerline

### Evaluation configuration


In [18]:
PREDICTION_RLE_JSON = (
    output_paths.inference_dir(RUN_NAME)
    / "predicted_annotations_rle.json"
)

EVALUATION_DIR = (
    output_paths.run_dir(RUN_NAME)
    / "eval"
    / "instance_metrics_cldice"
)

print("Run:")
print(RUN_NAME)

print("\nGround truth:")
print(GT_JSON)

print("\nPredictions:")
print(PREDICTION_RLE_JSON)

print("\nEvaluation output:")
print(EVALUATION_DIR)

if not GT_JSON.exists():
    raise FileNotFoundError(
        f"Ground-truth RLE JSON does not exist: {GT_JSON}"
    )

if not PREDICTION_RLE_JSON.exists():
    raise FileNotFoundError(
        f"Prediction RLE JSON does not exist: {PREDICTION_RLE_JSON}"
    )


Run:
cntlib_notebook_inference_example

Ground truth:
C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\COCO_mask\stratified_seed_0\test\annotations_rle.json

Predictions:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\inference\predicted_annotations_rle.json

Evaluation output:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\eval\instance_metrics_cldice


### Run evaluation
Predictions are matched one-to-one to ground-truth CNT instances using
clDice with a matching threshold of 0.5.
No prediction-score filtering or endpoint-based filtering is applied in
this example.

In [19]:
evaluation_args = [
    "--matching-metrics",
    "cldice",

    "--gt-rle-json",
    str(GT_JSON),

    "--pred-rle-json",
    str(PREDICTION_RLE_JSON),

    "--output-dir",
    str(EVALUATION_DIR),

    "--prediction-score-threshold",
    "0.0",

    "--cldice-thresholds",
    "0.5",
]

evaluation_args

['--matching-metrics',
 'cldice',
 '--gt-rle-json',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\data\\cnt_segmentation\\COCO_mask\\stratified_seed_0\\test\\annotations_rle.json',
 '--pred-rle-json',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\global_outputs\\runs\\cntlib_notebook_inference_example\\inference\\predicted_annotations_rle.json',
 '--output-dir',
 'C:\\Users\\abd93000\\PycharmProjects\\cnt_project_review\\global_outputs\\runs\\cntlib_notebook_inference_example\\eval\\instance_metrics_cldice',
 '--prediction-score-threshold',
 '0.0',
 '--cldice-thresholds',
 '0.5']

In [20]:
run_module(
    "cnt_project.evaluation.runners.eval_instance_metrics_runner",
    *evaluation_args,
)

Running:
c:\Users\abd93000\PycharmProjects\cnt_project_review\.venv_review\Scripts\python.exe -u -m cnt_project.evaluation.runners.eval_instance_metrics_runner --matching-metrics cldice --gt-rle-json C:\Users\abd93000\PycharmProjects\cnt_project_review\data\cnt_segmentation\COCO_mask\stratified_seed_0\test\annotations_rle.json --pred-rle-json C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\inference\predicted_annotations_rle.json --output-dir C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\eval\instance_metrics_cldice --prediction-score-threshold 0.0 --cldice-thresholds 0.5
--------------------------------------------------------------------------------
[1/30] 1293-22_3_0_20230306224436.tif: GT=96, predictions=154; endpoint filter disabled
[2/30] 400-1093-w11-c1p1-befo3-10sp-15flow_right.tif: GT=167, predictions=186; endpoint filter disabled
[3/30] 400-1093-w17-c1p3-20s

## Output Locations

When NanoVision is used from the repository workflows, inference and evaluation
artifacts are organized under the configured run/output structure.

Typical artifacts include:

- polygon prediction COCO JSON;
- instance-mask COCO JSON;
- evaluation CSV files;
- diagnostic outputs.

In [21]:
RUN_ROOT = output_paths.run_dir(
    RUN_NAME
)

INFERENCE_DIR = output_paths.inference_dir(
    RUN_NAME
)

EVALUATION_DIR = RUN_ROOT / "eval"
VISUALIZATION_DIR = RUN_ROOT / "viz"

print("Run root:")
print(RUN_ROOT)

print("\nInference outputs:")
print(INFERENCE_DIR)

print("\nEvaluation outputs:")
print(EVALUATION_DIR)

print("\nVisualization outputs:")
print(VISUALIZATION_DIR)
print("\nGenerated files:")

for path in sorted(RUN_ROOT.rglob("*")):
    if path.is_file():
        print(
            " -",
            path.relative_to(RUN_ROOT),
        )

Run root:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example

Inference outputs:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\inference

Evaluation outputs:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\eval

Visualization outputs:
C:\Users\abd93000\PycharmProjects\cnt_project_review\global_outputs\runs\cntlib_notebook_inference_example\viz

Generated files:
 - eval\instance_metrics_cldice\cldice_pair_visualizations\threshold_0p50\1293-22_3_0_20230306224436\pair_0001_gt_0000_pred_0139_cldice_0.774.png
 - eval\instance_metrics_cldice\cldice_pair_visualizations\threshold_0p50\1293-22_3_0_20230306224436\pair_0001_gt_0000_pred_0139_cldice_0.774.svg
 - eval\instance_metrics_cldice\cldice_pair_visualizations\threshold_0p50\1293-22_3_0_20230306224436\pair_0001_gt_0001_pred_0222_cldice_0.560.png
 - eval\instance_metri